# 🌱 Plant Watering AI — подготовка, нормализация, дерево решений
**Один источник:** `jibrand/Plant-dataset`. Числовые значения ниже **синтезированы** для учебной модели — это не реальные данные датчиков.

In [ ]:
%pip -q install pandas numpy scikit-learn joblib matplotlib seaborn huggingface_hub fsspec

## 1. Загрузка исходного датасета

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, joblib, json
from pathlib import Path
try:
    df = pd.read_csv("hf://datasets/jibrand/Plant-dataset/plant_dataset.csv")
except Exception as e:
    print("Загрузка с Hugging Face недоступна:", e)
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))
print("Размер исходного датасета:",df.shape)
display(df.head()); display(df.isna().sum().sort_values(ascending=False))

## 2. Строгая очистка и категориальная нормализация
Некоторые строки датасета имеют смещённые поля. Оставляем только распознанные значения света и полива.

In [ ]:
WATER_MAP={'moderate': 'Умеренный', 'regular': 'Регулярный', 'regularly': 'Регулярный', 'water regularly': 'Регулярный', 'infrequent': 'Редкий', 'infrequently': 'Редкий', 'water sparingly': 'Редкий', 'water moderately': 'Умеренный'}
LIGHT_MAP={'full sun': 'Полное солнце', 'full sun to partial shade': 'Солнце / полутень', 'partial shade to full sun': 'Солнце / полутень', 'partial shade': 'Полутень', 'bright indirect light': 'Яркий рассеянный', 'low light': 'Слабое освещение', 'shade': 'Тень'}
def clean(s):
    return s.fillna('').astype(str).str.strip().str.strip('"').str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
catalog=pd.DataFrame({"plant":df["Plant"].fillna("").astype(str).str.strip(),"light":clean(df["Light"]).map(LIGHT_MAP),"base_need":clean(df["Watering"]).map(WATER_MAP)}).dropna().drop_duplicates().reset_index(drop=True)
print("После очистки:",catalog.shape,"растений:",catalog.plant.nunique())
display(catalog.head())

## 3. Создание ЧИСЛОВЫХ сценариев и меток по демонстрационной логике
Пороги не получены измерением и не имеют статуса агрономических норм. Они нужны лишь для демонстрации обучения модели, зависящей от ползунков.

In [ ]:
def rule_label(base_need,soil,temp,humidity):
    threshold={"Редкий":22,"Умеренный":34,"Регулярный":45}[base_need]
    threshold+=5 if temp>=30 else (-5 if temp<15 else 0)
    threshold+=4 if humidity<40 else (-3 if humidity>75 else 0)
    return int(soil<threshold)
rng=np.random.default_rng(42)
blocks=[]
for _,rec in catalog.iterrows():
    n=24
    b=pd.DataFrame({"plant":[rec.plant]*n,"light":[rec.light]*n,"base_need":[rec.base_need]*n,"soil_moisture":rng.integers(5,96,n),"air_temperature":rng.integers(5,43,n),"air_humidity":rng.integers(15,96,n)})
    b["needs_watering"]=[rule_label(rec.base_need,int(s),int(t),int(h)) for s,t,h in zip(b.soil_moisture,b.air_temperature,b.air_humidity)]
    blocks.append(b)
simulated=pd.concat(blocks,ignore_index=True)
display(simulated.head()); print(simulated.needs_watering.value_counts())

## 4. Нормализация числовых признаков — MinMaxScaler
Для дерева решений масштабирование необязательно, но показываем его отдельно в учебных целях. Дерево обучаем на исходном масштабе, чтобы пороги оставались интерпретируемыми.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay, classification_report
FEATURES=["base_need","light","soil_moisture","air_temperature","air_humidity"]
X_train,X_test,y_train,y_test=train_test_split(simulated[FEATURES],simulated.needs_watering,test_size=.25,random_state=42,stratify=simulated.needs_watering)
num=["soil_moisture","air_temperature","air_humidity"]
scaler=MinMaxScaler().fit(X_train[num])
normalized_train=pd.DataFrame(scaler.transform(X_train[num]),columns=num)
display(normalized_train.head())
print("Нормализация рассчитана только по train — без утечки из test.")

## 5. Обучение Decision Tree (признаки действительно включают значения ползунков)

In [ ]:
prep=ColumnTransformer([("categorical",OneHotEncoder(handle_unknown="ignore"),["base_need","light"])],remainder="passthrough")
model=Pipeline([("prep",prep),("tree",DecisionTreeClassifier(max_depth=9,min_samples_leaf=8,random_state=42))])
model.fit(X_train,y_train)
pred=model.predict(X_test)
print("Accuracy (только синтетический тест):",accuracy_score(y_test,pred))
print("Balanced accuracy:",balanced_accuracy_score(y_test,pred))
print(classification_report(y_test,pred))
ConfusionMatrixDisplay.from_predictions(y_test,pred); plt.title("Матрица ошибок на синтетических сценариях"); plt.show()

## 6. Визуализация дерева и изменение показаний

In [ ]:
plt.figure(figsize=(19,8))
plot_tree(model.named_steps["tree"],max_depth=3,filled=True,fontsize=8)
plt.title("Верхние уровни дерева решений");plt.show()
example=pd.DataFrame([{"base_need":"Умеренный","light":"Полное солнце","soil_moisture":20,"air_temperature":31,"air_humidity":30}])
print("Тестовое показание:"); display(example)
print("Прогноз (1=полить, 0=не поливать):",model.predict(example)[0])
example["soil_moisture"]=80
print("После повышения влажности почвы:",model.predict(example)[0])

## 7. Экспорт для Streamlit
Перенесите файлы в соответствующие папки GitHub-проекта.

In [ ]:
from google.colab import files
Path("model").mkdir(exist_ok=True); Path("data").mkdir(exist_ok=True)
joblib.dump(model,"model/irrigation_tree.joblib")
catalog.to_csv("data/plant_catalog.csv",index=False)
simulated.to_csv("data/synthetic_training.csv",index=False)
metrics={"source":"jibrand/Plant-dataset ONLY","original_rows":len(df),"clean_rows":len(catalog),"synthetic_rows":len(simulated),"plants":catalog.plant.nunique(),"train_rows":len(X_train),"test_rows":len(X_test),"accuracy":round(float(accuracy_score(y_test,pred)),4),"balanced_accuracy":round(float(balanced_accuracy_score(y_test,pred)),4),"warning":"TEST LABELS SYNTHETIC: not validated on actual irrigation outcomes."}
Path("model/metrics.json").write_text(json.dumps(metrics,ensure_ascii=False,indent=2))
import shutil
shutil.make_archive("colab_trained_files","zip",".","model")
files.download("model/irrigation_tree.joblib")
files.download("data/plant_catalog.csv")
files.download("model/metrics.json")